## Project About
This project investigates whether GPTQ quantization at INT8 preserves model
quality consistently across two different LLM architectures
(Qwen2.5-1.5B-Instruct and SmolLM2-1.7B-Instruct). We evaluate quality using
perplexity on WikiText-2. Per-layer reconstruction error and lower bit-widths
(INT4, INT3) are identified as natural extensions, left for future work.

## Step 1: Install and Import
Installing dependencies and importing all required libraries

In [48]:
import logging
logging.disable(logging.WARNING)

In [49]:
# importing required libraries
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

In [50]:
# checking for gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Step 2: Load Baseline Models
Loading both models in FP16 to establish our baseline before any quantization
is applied. Qwen2.5-1.5B-Instruct and SmolLM2-1.7B-Instruct represent two
different architecture families, which lets us later check whether
quantization degradation patterns are consistent across them or specific to
one model. Both are openly available on Hugging Face with no gating or
access token required.

In [51]:
first_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model_qwen = AutoModelForCausalLM.from_pretrained(
    first_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [52]:
second_model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

model_smol = AutoModelForCausalLM.from_pretrained(
    second_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

## Step 3: Sanity Check Baseline Models
Loading tokenizers for both models and running a simple test prompt through
each to confirm they produce coherent output before we introduce any
quantization complexity. We'll also record GPU memory usage for each model
in FP16 as a reference point to compare against later once quantized
versions are loaded.

In [53]:
# loading tokenizers for both models
tokenizer_qwen = AutoTokenizer.from_pretrained(first_model_name)
tokenizer_smol = AutoTokenizer.from_pretrained(second_model_name)

prompt = "Give me a short introduction about Linked List" # declaring prompt
messages = [
    {"role":"system","content":"You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

text_qwen = tokenizer_qwen.apply_chat_template(  # applying chat template
    messages,
    tokenize=False,
    add_generation_prompt=True
)

text_smol = tokenizer_smol.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs_qwen = tokenizer_qwen(
    [text_qwen],
    return_tensors="pt"
).to(device)

model_inputs_smol = tokenizer_smol(
    [text_smol],
    return_tensors="pt"
).to(device)

# now generated ids for both the models
# check
generated_ids_qwen = model_qwen.generate(
    **model_inputs_qwen,
    max_new_tokens=200,
)

generated_ids_smol = model_smol.generate(
    **model_inputs_smol,
    max_new_tokens=200,
)

In [54]:
# now checking the responses
# for model qwen
generated_ids_qwen = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs_qwen.input_ids, generated_ids_qwen)
]

response = tokenizer_qwen.batch_decode(generated_ids_qwen, skip_special_tokens=True)[0]
print(response)

A linked list is a linear data structure in which each element (or node) contains two parts: a value and a reference to the next node in the sequence. The last node's reference points back to the first node, forming a cycle.
Linked lists can be implemented using either an array or pointers. In an array implementation, elements are stored contiguously in memory with their addresses stored as references within the nodes. In a pointer-based implementation, each node stores only its own address rather than an entire block of memory.
Linked lists have several advantages over arrays, including dynamic resizing, efficient insertion and deletion at any position in the list, and support for random access. However, they also suffer from increased space overhead due to the additional pointers required per node.


In [55]:
# for now model smol
generated_ids_smol = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs_smol.input_ids, generated_ids_smol)
]

response = tokenizer_smol.batch_decode(generated_ids_smol, skip_special_tokens=True)[0]
print(response)

A linked list is a linear data structure where elements are not stored in contiguous memory locations. Instead, each element, known as a node, contains a reference (or "link") to the next node in the sequence. This allows for efficient insertion or removal of elements at any position in the sequence. Linked lists are commonly used in applications where data is frequently added or removed, such as in web browsers, where new web pages are constantly being loaded and removed.


## Step 4: Prepare Calibration and Evaluation Data
Loading WikiText-2 and splitting its role in two: a small sample from the
train split will serve as GPTQ's calibration data (used to compute the
Hessian/importance scores during quantization), while the test split is
reserved purely for perplexity evaluation later. Keeping these two
completely separate avoids data leakage between calibration and evaluation.

In [56]:
# loading the wikitext2 dataset
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


## Step 5: Build Calibration Set
Selecting a small sample of text from the WikiText-2 train split to serve
as GPTQ's calibration data. We filter out empty lines (WikiText-2 has many
blank entries from its raw formatting) and select a fixed number of
non-empty text samples, tokenized separately for each model's tokenizer
since Qwen and SmolLM2 use different vocabularies.

In [57]:
# dataset is splitted into train,validation and test.
# checking some random rows of training data
for row in range(20,25):
  print(f"Row {row} : {dataset['train'][row]['text']}")

Row 20 : 
Row 21 :  Concept work for Valkyria Chronicles III began after development finished on Valkyria Chronicles II in early 2010 , with full development beginning shortly after this . The director of Valkyria Chronicles II , Takeshi Ozawa , returned to that role for Valkyria Chronicles III . Development work took approximately one year . After the release of Valkyria Chronicles II , the staff took a look at both the popular response for the game and what they wanted to do next for the series . Like its predecessor , Valkyria Chronicles III was developed for PlayStation Portable : this was due to the team wanting to refine the mechanics created for Valkyria Chronicles II , and they had not come up with the " revolutionary " idea that would warrant a new entry for the PlayStation 3 . Speaking in an interview , it was stated that the development team considered Valkyria Chronicles III to be the series ' first true sequel : while Valkyria Chronicles II had required a large amount of t

In [58]:
# counting the total blank entries
count = 0

for row in range(len(dataset['train'])):
  if dataset['train'][row]['text']=="":
    count+=1

print(count)

12951


In [59]:
# storing valid rows in a list
# which does not contain empty list and does not have "=== Music ===" like heading or structure
valid_rows = []

for row in range(len(dataset['train'])):
  str1 = dataset['train'][row]['text'].strip()

  if str1!="" and not str1.startswith("="):
    valid_rows.append(str1)

# final text
final_text = "\n\n".join(valid_rows)
print(final_text[:200]) # just for sanity check

Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ pl


## Step 6: Chunk Text into Calibration Samples
Splitting the filtered text into fixed-length token chunks, then selecting
128 of them as GPTQ's calibration samples. Each chunk is tokenized using
the target model's own tokenizer, since Qwen and SmolLM2 have different
vocabularies — meaning we'll produce two separate calibration sets, one
per model. A common calibration sequence length is 512 tokens per sample.

In [60]:
# chunking
# text into calibration sample
# for model Qwen
chunk_size = 512
num_samples = 128

# turning the entire string into a list of token ids
all_tokens_qwen = tokenizer_qwen.encode(final_text,add_special_tokens=False)

# now slicing the token list into chunk size of 512
chunks = [] # storing in list
for i in range(0,len(all_tokens_qwen),chunk_size):
  chunk = all_tokens_qwen[i:i+chunk_size]
  if len(chunk)==chunk_size: # dropping any chunk which has a length less than 512
    chunks.append(chunk)

# calibration sample for first model
calibration_sample_qwen = torch.tensor(chunks[:num_samples])

In [61]:
# shape of the calibration_sample_qwen
print(calibration_sample_qwen.shape)

torch.Size([128, 512])


In [62]:
# now for the second model
# turning the entire string into a list of token ids

all_tokens_smol = tokenizer_smol.encode(final_text,add_special_tokens=False)
# now slicing the token list into chunk size of 512
chunks_smol = [] # storing in list
for i in range(0,len(all_tokens_smol),chunk_size):
  chunk = all_tokens_smol[i:i+chunk_size]
  if len(chunk)==chunk_size: # dropping any chunk which has a length less than 512
    chunks_smol.append(chunk)

# calibration sample for first model
calibration_sample_smol = torch.tensor(chunks_smol[:num_samples])

In [63]:
# shape of the second sample
print(calibration_sample_smol.shape)

torch.Size([128, 512])


## Step 7: Reformat Calibration Data for GPTQModel
GPTQModel expects calibration data as a list of dictionaries, each with
"input_ids" and "attention_mask" keys, rather than a raw tensor. We convert
our existing [128, 512] calibration tensors into this format for both
models - since every chunk is exactly 512 tokens with no padding, each
attention_mask is simply a tensor of all 1s matching that length.

In [64]:
# here no padding is used
# So, in attention mask, all will be one's
attention_mask = torch.tensor([1]*512)
# list
calibration_data_qwen = []

for row in calibration_sample_qwen:
    sample_dict = {} # declaring dictionary
    sample_dict['input_ids'] = row
    sample_dict['attention_mask'] = attention_mask
    # now appending into the list
    calibration_data_qwen.append(sample_dict)

print(calibration_data_qwen[0])

{'input_ids': tensor([ 19617,     73,  55661,    902,  85162,     88,   4204,    220,     18,
           549,   1230,   8548,    291,  65316,    320,  10769,    549,  49434,
            99,  74167,  15767, 139330,  32610, 135809, 127176,     18,   1154,
         13020,    659,  85162,     88,   4204,    315,    279,  70635,    220,
            18,    873,   1154,  16626,  13862,    311,    438,  85162,     88,
          4204,  65316,  14429,   4889,   6323,   1154,    374,    264,  38647,
          3476,    569,     12,     31,   5619,   2766,   1809,   7881,    553,
         79849,    323,   7816,   5058,   1816,    369,    279,  31265,  41485,
           659,  44794,    304,   6058,    220,     17,     15,     16,     16,
           304,   6323,   1154,    432,    374,    279,   4843,   1809,    304,
           279,  85162,     88,   4204,   4013,    659,  20782,    287,    279,
          1852,  36508,    315,  38647,    323,   1931,    569,     12,     31,
           882,  26029,   

In [65]:
# length of calibration_data_qwen
print(len(calibration_data_qwen))

128


In [66]:
# now for second model# here no padding is used
# So, in attention mask, all will be one's
attention_mask_smol = torch.tensor([1]*512)
# list
calibration_data_smol = []

for row in calibration_sample_smol:
    sample_dict = {} # declaring dictionary
    sample_dict['input_ids'] = row
    sample_dict['attention_mask'] = attention_mask_smol
    # now appending into the list
    calibration_data_smol.append(sample_dict)

print(len(calibration_data_smol)) # length

128


## Step 8: Configure and Run GPTQ Quantization
Defining QuantizeConfig for INT8, loading each model through GPTQModel,
and running model.quantize() using the calibration data prepared in the
previous step. This is where GPTQModel internally computes the Hessian
from calibration inputs and performs the column-by-column, error-
compensated quantization for every Linear layer in the model. Each
quantized model is saved separately for evaluation.

In [67]:
from transformers.utils.hub import hf_api

print("hf_api works")

hf_api works


In [68]:
# loading required libraries
from gptqmodel import GPTQModel, QuantizeConfig
print("GPTQModel imported successfully!")

GPTQModel imported successfully!


In [69]:
# confguring for 8 bit quantization
quantize_config_8bit = QuantizeConfig(
    bits=8,
    group_size=128,
    sym=True,
    desc_act=False,
)

In [70]:
# loading QWEN and running INT8 quantization
model_qwen_gptq_8bit = GPTQModel.load(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantize_config=quantize_config_8bit,
)

# running quantization using prepared calibration data
model_qwen_gptq_8bit.quantize(calibration_data_qwen,batch_size=1)
# saving the model
model_qwen_gptq_8bit.save("qwen-int8")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Writing model shards: 0it [00:00, ?it/s]

Files in directory:
model.safetensors
quantize_config.json
chat_template.jinja
quant_log.csv
tokenizer_config.json
tokenizer.json
generation_config.json
config.json
Content of saved `generation_config.json`:
{
    "bos_token_id": 151643,
    "do_sample": true,
    "eos_token_id": [
        151645,
        151643
    ],
    "pad_token_id": 151643,
    "repetition_penalty": 1.1,
    "temperature": 0.7,
    "top_k": 20,
    "top_p": 0.8,
    "transformers_version": "5.14.1"
}
Content of saved `config.json`:
{
    "architectures": [
        "Qwen2ForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 151643,
    "dtype": "bfloat16",
    "eos_token_id": 151645,
    "hidden_act": "silu",
    "hidden_size": 1536,
    "initializer_range": 0.02,
    "intermediate_size": 8960,
    "layer_types": [
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
    

In [71]:
del model_qwen_gptq_8bit
import gc
gc.collect()
torch.cuda.empty_cache()

In [72]:
# loading second model smol and quantizing 8 bit
# loading Qwen and running INT8 quantization
model_smol_gptq_8bit = GPTQModel.load(
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    quantize_config_8bit
)

# running quantization using prepared calibration data
model_smol_gptq_8bit.quantize(calibration_data_smol, batch_size=1)
# saving the model
model_smol_gptq_8bit.save("smol-int8")

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

Writing model shards: 0it [00:00, ?it/s]

Files in directory:
model.safetensors
quantize_config.json
chat_template.jinja
quant_log.csv
tokenizer_config.json
tokenizer.json
generation_config.json
config.json
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 1,
    "do_sample": true,
    "eos_token_id": 2,
    "pad_token_id": 2,
    "transformers_version": "5.14.1"
}
Content of saved `config.json`:
{
    "architectures": [
        "LlamaForCausalLM"
    ],
    "attention_bias": false,
    "attention_dropout": 0.0,
    "bos_token_id": 1,
    "dtype": "bfloat16",
    "eos_token_id": 2,
    "head_dim": 64,
    "hidden_act": "silu",
    "hidden_size": 2048,
    "initializer_range": 0.02,
    "intermediate_size": 8192,
    "max_position_embeddings": 8192,
    "mlp_bias": false,
    "model_type": "llama",
    "num_attention_heads": 32,
    "num_hidden_layers": 24,
    "num_key_value_heads": 32,
    "pad_token_id": 2,
    "pretraining_tp": 1,
    "quantization_config": {
        "bits": 8,

In [73]:
del model_smol_gptq_8bit
gc.collect()
torch.cuda.empty_cache()

## Step 9: Prepare Evaluation Data
Loading the WikiText-2 test split (completely separate from the train split
used for calibration) and preparing it the same way - filtering out blank
lines and section headers, then tokenizing and chunking into fixed-length,
non-overlapping sequences. This held-out set will be used to compute
perplexity for all four model versions, and is never touched during
calibration or quantization.

In [74]:
# storing valid rows in a lost
# which does not contain empty list and does not have "=== Music ===" like heading or structure
test_valid_rows = []

for row in range(len(dataset['test'])):
  str1 = dataset['test'][row]['text'].strip()

  if str1!="" and not str1.startswith("="):
    test_valid_rows.append(str1)

# final text
test_final_text = "\n\n".join(test_valid_rows)
print(test_final_text[:200]) # just for sanity check

Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons 


In [75]:
# chunking
# preparing data for testing
chunk_size = 512

# turning the entire string into a list of token ids
all_test_tokens_qwen = tokenizer_qwen.encode(test_final_text,add_special_tokens=False)

# now slicing the token list into chunk size of 512
test_chunks = [] # storing in list
for i in range(0,len(all_test_tokens_qwen),chunk_size):
  chunk = all_test_tokens_qwen[i:i+chunk_size]
  if len(chunk)==chunk_size: # dropping any chunk which has a length less than 512
    test_chunks.append(chunk)

# test-sample
test_sample_qwen = torch.tensor(test_chunks)

In [76]:
# shape of sample data
print(test_sample_qwen.shape)

torch.Size([567, 512])


In [77]:
# now for smol model
# chunking eval text for SmolLM2
all_test_tokens_smol = tokenizer_smol.encode(test_final_text, add_special_tokens=False)

test_chunks_smol = []
for i in range(0, len(all_test_tokens_smol), chunk_size):
    chunk = all_test_tokens_smol[i:i+chunk_size]
    if len(chunk) == chunk_size:
        test_chunks_smol.append(chunk)

test_sample_smol = torch.tensor(test_chunks_smol)
print(test_sample_smol.shape)

torch.Size([580, 512])


## Step 10: Load All Models for Evaluation
Loading the two FP16 baselines (Qwen, SmolLM2) and the two INT8 quantized
models (saved from GPTQModel) back into memory, so all four versions are
ready for perplexity and per-layer reconstruction error evaluation. We
load these one at a time and are careful about memory, since we'll need
multiple models loaded simultaneously to compare their outputs.

## Step 11: Compute Perplexity for All Four Model Versions
Loading one model at a time, running it over its eval chunks, and computing token-count-
weighted average cross-entropy loss, converted to perplexity. We do this
for Qwen FP16, Qwen INT8, SmolLM2 FP16, and SmolLM2 INT8 - deleting each
model from memory before loading the next, so only one model is ever
loaded at a time.

In [78]:
import math

# load Qwen FP16
model_qwen = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
model_qwen.eval()

total_loss = 0.0
num_chunks = test_sample_qwen.shape[0]

with torch.no_grad():
    for i in range(num_chunks):
        input_ids = test_sample_qwen[i].unsqueeze(0).to(device)  # shape [1, 512]

        outputs = model_qwen(input_ids, labels=input_ids)
        loss = outputs.loss.item()

        total_loss += loss

        if i % 50 == 0:
            print(f"chunk {i}/{num_chunks}, running avg loss: {total_loss/(i+1):.4f}")

avg_loss_qwen_fp16 = total_loss / num_chunks
perplexity_qwen_fp16 = math.exp(avg_loss_qwen_fp16)

print(f"\nQwen FP16 — avg loss: {avg_loss_qwen_fp16:.4f}, perplexity: {perplexity_qwen_fp16:.4f}")

# cleanup before loading next model
del model_qwen
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

chunk 0/567, running avg loss: 2.1284
chunk 50/567, running avg loss: 2.4432
chunk 100/567, running avg loss: 2.5158
chunk 150/567, running avg loss: 2.4657
chunk 200/567, running avg loss: 2.4960
chunk 250/567, running avg loss: 2.4813
chunk 300/567, running avg loss: 2.4847
chunk 350/567, running avg loss: 2.4789
chunk 400/567, running avg loss: 2.5178
chunk 450/567, running avg loss: 2.5132
chunk 500/567, running avg loss: 2.5119
chunk 550/567, running avg loss: 2.5084

Qwen FP16 — avg loss: 2.5116, perplexity: 12.3246


In [79]:
# load Qwen INT8 (from disk, already quantized)
model_qwen_int8 = GPTQModel.load("qwen-int8", backend="torch")
model_qwen_int8.eval()

total_loss = 0.0
num_chunks = test_sample_qwen.shape[0]

with torch.no_grad():
    for i in range(num_chunks):
        input_ids = test_sample_qwen[i].unsqueeze(0).to(device)

        outputs = model_qwen_int8(input_ids, labels=input_ids)
        loss = outputs.loss.item()

        total_loss += loss

        if i % 50 == 0:
            print(f"chunk {i}/{num_chunks}, running avg loss: {total_loss/(i+1):.4f}")

avg_loss_qwen_int8 = total_loss / num_chunks
perplexity_qwen_int8 = math.exp(avg_loss_qwen_int8)

print(f"\nQwen INT8 — avg loss: {avg_loss_qwen_int8:.4f}, perplexity: {perplexity_qwen_int8:.4f}")

# cleanup before loading next model
del model_qwen_int8
gc.collect()
torch.cuda.empty_cache()

from_quantized: adapter: None


  0%|          | 0/926 [00:00<?, ?w/s]

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:1946: UserWarning: Dynamo does not know how to trace the builtin `triton._C.libtriton.getenv_bool.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
I

chunk 0/567, running avg loss: 2.1328
chunk 50/567, running avg loss: 2.4440
chunk 100/567, running avg loss: 2.5165
chunk 150/567, running avg loss: 2.4663
chunk 200/567, running avg loss: 2.4968
chunk 250/567, running avg loss: 2.4821
chunk 300/567, running avg loss: 2.4855
chunk 350/567, running avg loss: 2.4797
chunk 400/567, running avg loss: 2.5186
chunk 450/567, running avg loss: 2.5141
chunk 500/567, running avg loss: 2.5128
chunk 550/567, running avg loss: 2.5093

Qwen INT8 — avg loss: 2.5125, perplexity: 12.3352


In [80]:
# load SmolLM2 FP16
model_smol = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
model_smol.eval()

total_loss = 0.0
num_chunks = test_sample_smol.shape[0]

with torch.no_grad():
    for i in range(num_chunks):
        input_ids = test_sample_smol[i].unsqueeze(0).to(device)

        outputs = model_smol(input_ids, labels=input_ids)
        loss = outputs.loss.item()

        total_loss += loss

        if i % 50 == 0:
            print(f"chunk {i}/{num_chunks}, running avg loss: {total_loss/(i+1):.4f}")

avg_loss_smol_fp16 = total_loss / num_chunks
perplexity_smol_fp16 = math.exp(avg_loss_smol_fp16)

print(f"\nSmolLM2 FP16 — avg loss: {avg_loss_smol_fp16:.4f}, perplexity: {perplexity_smol_fp16:.4f}")

# cleanup before loading next model
del model_smol
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

chunk 0/580, running avg loss: 2.0856
chunk 50/580, running avg loss: 2.3976
chunk 100/580, running avg loss: 2.4519
chunk 150/580, running avg loss: 2.4185
chunk 200/580, running avg loss: 2.4489
chunk 250/580, running avg loss: 2.4397
chunk 300/580, running avg loss: 2.4205
chunk 350/580, running avg loss: 2.4162
chunk 400/580, running avg loss: 2.4549
chunk 450/580, running avg loss: 2.4382
chunk 500/580, running avg loss: 2.4414
chunk 550/580, running avg loss: 2.4360

SmolLM2 FP16 — avg loss: 2.4421, perplexity: 11.4971


In [81]:
# load SmolLM2 INT8 (from disk, already quantized)
model_smol_int8 = GPTQModel.load("smol-int8", backend="torch")
model_smol_int8.eval()

total_loss = 0.0
num_chunks = test_sample_smol.shape[0]

with torch.no_grad():
    for i in range(num_chunks):
        input_ids = test_sample_smol[i].unsqueeze(0).to(device)

        outputs = model_smol_int8(input_ids, labels=input_ids)
        loss = outputs.loss.item()

        total_loss += loss

        if i % 50 == 0:
            print(f"chunk {i}/{num_chunks}, running avg loss: {total_loss/(i+1):.4f}")

avg_loss_smol_int8 = total_loss / num_chunks
perplexity_smol_int8 = math.exp(avg_loss_smol_int8)

print(f"\nSmolLM2 INT8 — avg loss: {avg_loss_smol_int8:.4f}, perplexity: {perplexity_smol_int8:.4f}")

# cleanup
del model_smol_int8
gc.collect()
torch.cuda.empty_cache()

from_quantized: adapter: None


  0%|          | 0/722 [00:00<?, ?w/s]

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:1946: UserWarning: Dynamo does not know how to trace the builtin `triton._C.libtriton.getenv_bool.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
I

chunk 0/580, running avg loss: 2.0867
chunk 50/580, running avg loss: 2.3981
chunk 100/580, running avg loss: 2.4523
chunk 150/580, running avg loss: 2.4185
chunk 200/580, running avg loss: 2.4489
chunk 250/580, running avg loss: 2.4395
chunk 300/580, running avg loss: 2.4203
chunk 350/580, running avg loss: 2.4159
chunk 400/580, running avg loss: 2.4547
chunk 450/580, running avg loss: 2.4381
chunk 500/580, running avg loss: 2.4414
chunk 550/580, running avg loss: 2.4358

SmolLM2 INT8 — avg loss: 2.4420, perplexity: 11.4960
